# 🏦 Handout: Text-to-Dataset with `LLMSynthesizer`

In this handout, you will learn how to generate a structured and relational dataset **from a written description**, no source data required.

You declare the schema in plain language: tables, columns, types, allowed values, keys and relationships. `LLMSynthesizer` handles the rest: batching, key allocation and validation, so the model writes the *content* of the fields while deterministic code holds the *structure* together.

In this notebook, we will:
1. Describe a two-table retail banking schema
2. Generate the data and inspect both tables

In [ ]:
subscription_key='add-your-subscription-key' #Use openai key or your anthropic api key


In [ ]:
from ydata.synthesizers.llm import LLMSynthesizer

synth = LLMSynthesizer(subscription_key=subscription_key)

print('LLMSynthesizer ready ✓')

## 1. Describe the Schema

Two tables: `customers` and their `transactions`.

Each column carries a `prompt` — what the value means — and a `dtype`. Categorical columns list their allowed `values`, so the model can only pick from that set. `primary_key` and `foreign_keys` are what turn two independent tables into a relational dataset: the SDK allocates the keys itself, in code, rather than asking the model to invent them.

In [ ]:
## Get a schema from a text description
from ydata.synthesizers.llm.schema_from_prompt import generate_schema

tables = generate_schema(
    description="""
    I want a database that represents a company that sells products to customers in the US. The company has products in the categories of food, drinks and snacks. The company sells its products through multiple channels such as online and in-store.
    The bases must include 2 tables:
    - Customers: basic information about the customers
    - Transactions: information about the transactions made by the customers
    """
)

## Or
## Set your schema manually with the desired tables, columns and data types
tables = {
    'customers': {
        'prompt': 'Customers of a US company that sells food, drinks and snacks',
        'columns': {
            'customer_id':  {'prompt': 'unique identifier in format CUST-000001', 'dtype': 'string'},
            'name':         {'prompt': 'full name of the customer', 'dtype': 'string'},
            'email':        {'prompt': 'customer email address', 'dtype': 'string'},
            'city':         {'prompt': 'US city of residence', 'dtype': 'string'},
            'state':        {'prompt': 'two-letter US state code', 'dtype': 'string'},
            'signup_date':  {'prompt': 'date the customer first registered', 'dtype': 'date'},
        },
        'primary_key': 'customer_id',
    },
    'transactions': {
        'prompt': 'Purchases made by customers, online and in-store',
        'columns': {
            'transaction_id': {'prompt': 'unique identifier in format TXN-000001', 'dtype': 'string'},
            'customer_id':    {'prompt': 'the customer who made the purchase', 'dtype': 'string'},
            'date':           {'prompt': 'date of the purchase', 'dtype': 'date'},
            'product_name':   {'prompt': 'name of the product purchased, consistent with its category',
                               'dtype': 'string'},
            'category':       {'prompt': 'product category', 'dtype': 'category',
                               'values': ['food', 'drinks', 'snacks']},
            'channel':        {'prompt': 'sales channel used for the purchase', 'dtype': 'category',
                               'values': ['online', 'in-store']},
            'quantity':       {'prompt': 'number of units purchased, between 1 and 10', 'dtype': 'int'},
            'unit_price':     {'prompt': 'price per unit in USD, between 1 and 50', 'dtype': 'float'},
            'total_amount':   {'prompt': 'quantity multiplied by unit_price, in USD', 'dtype': 'float'},
        },
        'primary_key': 'transaction_id',
        'foreign_keys': [
            {'column': 'customer_id',
             'referenced_table': 'customers',
             'prompt': 'each customer has between 2 and 10 transactions'},
        ],
    },
}

print(f'Schema declared: {list(tables)} ✓')

## 2. Generate

`fit()` takes the schema, `sample()` produces the rows. `sample_size` can be a single number for every table, or a dict for per-table control.

In [ ]:
synth.fit(tables=tables)

## Define the sample size for the parent table
## In this case, we want 20 customers, where each customer has between 2 and 10 transactions as defined in the prompt for the transactions table FK.
data = synth.sample(sample_size=20)

print('Generation complete ✓')

## 3. Inspect the Output

A multi-table schema returns a `MultiDataset` — each table is accessible by name.

In [ ]:
print('Customers:')
print(data['customers'].head())

In [ ]:
print('Transactions:')
print(data['transactions'].head(10))

## ✅ Summary

In this lesson you:

1. **Declared a schema in plain language** — tables, columns, types, allowed values, a primary key and a foreign key
2. **Generated a relational dataset from it** with `fit()` and `sample()`, without a single row of source data

The key insight: the prompt describes *what* the data should look like, and the pipeline is what keeps it consistent at volume.